# Prefect & Workflow Engine — genai-tk Tutorial

This notebook demonstrates the two ways to run Prefect flows with genai-tk:

1. **Standard `@flow` functions** — write Python, call directly via the managed Prefect server.
2. **YAML-defined workflows** — declare pipelines in YAML, run with `cli workflow run` or the `flow_from_yaml` / `PrefectFlowFactory` helpers.

Both approaches use the same local Prefect server singleton managed by `cli prefect`.

> **Docs:** [prefect.md](../../docs/prefect.md) · [workflows.md](../../docs/workflows.md)


## 1. Setup & Imports

Start the local Prefect server (if not already running) and import the toolkit.


In [1]:
%load_ext autoreload
%autoreload 2
%reset -f


import sys

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv

In [ ]:
from genai_tk.utils.prefect_server import prefect_server

# Start the server (no-op if already running; auto-starts on workflow run if auto_start: true)
server = prefect_server()
server.ensure_running()
server.configure_api_url()  # sets PREFECT_API_URL so Prefect flows find the server

print("Prefect server running:", server.is_running())
print("API URL:", server.api_url)
print("UI URL:", server.ui_url)


Prefect server running: True
API URL: http://127.0.0.1:4200/api


In [ ]:
debug(server)

## 2. Prefect Basics: Declaring a `@flow`

Any standard Prefect `@flow` and `@task` works with the genai-tk provided Prefect server.
The server just needs to be running (done in section 1).


In [4]:
from prefect import flow, task


@task
def shout(word: str) -> str:
    """Convert a word to uppercase."""
    return word.upper()


@flow(name="hello-flow")
def hello_flow(words: list[str]) -> list[str]:
    """A minimal example @flow: shout each word in parallel."""
    futures = [shout.submit(w) for w in words]
    return [f.result() for f in futures]


# --- Execute the flow ---
# Because the server is running, this run will be recorded in the Prefect UI.
results = hello_flow(words=["hello", "prefect", "genai-tk"])
print("Results:", results)


20:58:54.793 | INFO    | Flow run 'pompous-mammoth' - Beginning flow run 'pompous-mammoth' for flow 'hello-flow'

20:58:54.823 | INFO    | Flow run 'pompous-mammoth' - View at http://127.0.0.1:4200/runs/flow-run/2c270c4a-1a91-4da6-b840-d6143c74b58c

20:58:54.891 | INFO    | Task run 'shout-769' - Finished in state Completed()

20:58:54.899 | INFO    | Task run 'shout-83d' - Finished in state Completed()

20:58:54.906 | INFO    | Task run 'shout-497' - Finished in state Completed()

20:58:55.858 | INFO    | Flow run 'pompous-mammoth' - Finished in state Completed()

Results: ['HELLO', 'PREFECT', 'GENAI-TK']


In [6]:
import os

# Open the Prefect UI to see the run we just triggered
prefect_ui = server.ui_url
print(f"Prefect UI: {prefect_ui}")
print("Open the URL above in your browser to inspect runs, tasks, and logs.")

Prefect UI: http://127.0.0.1:4200
Open the URL above in your browser to inspect runs, tasks, and logs.


## 3. Running Workflows via the CLI

The `cli workflow` command group is the standard way to run, inspect, and validate workflows.
Below we run CLI commands from the notebook using `subprocess` so you can see the output inline.


In [7]:
import subprocess, sys


def cli(*args: str) -> None:
    """Run a genai-tk CLI command and print the output."""
    cmd = ["uv", "run", "cli", *args]
    print("$ " + " ".join(cmd[2:]))  # show the user-facing command
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)


# List all available workflows
cli("workflow", "list")


$ cli workflow list
                                   Workflows                                    
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Workflow             ┃ Presets ┃ Steps ┃ Description                         ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ anonymize            │ docs    │     1 │ Detect and replace PII in text      │
│                      │         │       │ files (Presidio + Faker)            │
├──────────────────────┼─────────┼───────┼─────────────────────────────────────┤
│ anonymize_and_ingest │ docs    │     2 │ Anonymize text files then ingest    │
│                      │         │       │ into a RAG vector store             │
├──────────────────────┼─────────┼───────┼─────────────────────────────────────┤
│ baml_extract         │ default │     1 │ Extract structured data from files  │
│                      │         │       │ (Markdown/PDF) using BAML           │
├───────

In [8]:
# Inspect a specific workflow: DAG, parameters, presets
cli("workflow", "show", "markdownize")


$ cli workflow show markdownize
╭────────────────────────────────── Workflow ──────────────────────────────────╮
│ markdownize  Convert documents (PDF/DOCX/PPTX) to Markdown format            │
╰──────────────────────────────────────────────────────────────────────────────╯
                              Steps (single-step)                               
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ ID  ┃ Run                                                            ┃ Cache ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ run │ genai_tk.workflow.prefect.flows.markdownize_flow.markdownize_… │ none  │
└─────┴────────────────────────────────────────────────────────────────┴───────┘
╭────────────────────────────────── Defaults ──────────────────────────────────╮
│ {                                                                            │
│   "pdf_converter": "mistral",                                              

In [9]:
# Dry-run: resolve the plan without executing anything
# (Replace "markdownize" with any workflow name from the list above)
cli("workflow", "run", "markdownize", "--dry-run", "--base-dir", "/tmp/demo_in", "--to", "/tmp/demo_out")


$ cli workflow run markdownize --dry-run --base-dir /tmp/demo_in --to /tmp/demo_out
    Workflow Resolution    
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Property  ┃ Value       ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Requested │ markdownize │
│ Workflow  │ markdownize │
│ Preset    │ <none>      │
│ Force     │ False       │
│ Steps     │ 1           │
└───────────┴─────────────┘
╭────────────────────────────── Effective Values ──────────────────────────────╮
│ {                                                                            │
│   "pdf_converter": "mistral",                                                │
│   "batch_size": 5,                                                           │
│   "base_dir": "/tmp/demo_in",                                                │
│   "output_dir": "/tmp/demo_out",                                             │
│   "pathspecs": null                                                          │
│ }                                                                  

## 4. YAML-Defined Workflows: DSL Primer

The Workflow DSL lives in YAML files under `config/` (auto-scanned on startup).
Here is the anatomy of a workflow definition:

```yaml
workflows:
  my_pipeline:
    description: "Short human-readable description"

    # --- Single-step shorthand ---
    run: my_module.my_flow          # dotted Python path to a @flow or callable
    defaults:
      base_dir: /data/in
      output_dir: /data/out
    params:
      base_dir: {required: true}   # document required params
      output_dir: {required: true}
    presets:
      demo:                        # named value bundle: cli workflow run my_pipeline/demo
        base_dir: /tmp/demo
        output_dir: /tmp/demo_out

    # --- Multi-step pipeline (use pipeline: instead of run:) ---
    # pipeline:
    #   - id: step_a
    #     run: my_module.flow_a
    #     with:
    #       input_dir: "${values.base_dir}"
    #   - id: step_b
    #     run: my_module.flow_b
    #     after: [step_a]          # dependency: step_b waits for step_a
    #     with:
    #       source: "${values.base_dir}"
    #     cache: manifest          # skip if inputs unchanged
```

**Key rules:**
- Each workflow has either `run:` (single step) **or** `pipeline:` (multi-step) — not both.
- `defaults:` keys are auto-wired to the callable — no explicit `with:` needed for single-step.
- `presets:` live inside the workflow; select with `workflow_name/preset_name`.
- `after:` is the dependency list; independent steps run in parallel via `ThreadPoolTaskRunner`.
- `${values.key}` in `with:` resolves against `defaults → preset → CLI --set`.


## 5. Loading a YAML Workflow Programmatically

Use `load_workflows()` + `resolve_workflow_invocation()` + `WorkflowCompiler` to inspect a
workflow without running it.  This is the same chain the CLI uses internally.


In [10]:
from genai_tk.workflow import (
    load_workflows,
    resolve_workflow_invocation,
    WorkflowCompiler,
)

# 1. List all registered workflows (from config/ YAML files)
workflows = load_workflows()
print("Available workflows:", sorted(workflows.keys()))


Available workflows: ['anonymize', 'anonymize_and_ingest', 'baml_extract', 'baml_run', 'baml_to_table', 'json_to_table', 'markdownize', 'merge_markdown', 'ppt2pdf', 'rag_ingest']


In [11]:
# 2. Resolve a workflow invocation (merges defaults, preset, CLI overrides)
invocation = resolve_workflow_invocation(
    "markdownize",
    cli_overrides={"base_dir": "/tmp/demo_in", "output_dir": "/tmp/demo_out"},
)
print("Workflow name :", invocation.workflow_name)
print("Resolved values:", invocation.values)

# 3. Compile it (validates targets, topological sort, expands sub-workflows)
compiled = WorkflowCompiler().compile(invocation.workflow, invocation.values)
print("\nCompiled steps:")
for step in compiled.steps:
    target = step.invoke.target if step.invoke else "-"
    print(f"  [{step.id}]  →  {target}  (wait_for={step.wait_for})")


2026-05-30 21:03:07.324 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'markdownize': 1 steps


Workflow name : markdownize
Resolved values: {'pdf_converter': 'mistral', 'batch_size': 5, 'base_dir': '/tmp/demo_in', 'output_dir': '/tmp/demo_out', 'pathspecs': None}

Compiled steps:
  [run]  →  genai_tk.workflow.prefect.flows.markdownize_flow.markdownize_flow  (wait_for=[])


## 6. `flow_from_yaml` — Inline YAML to Prefect Flow

`flow_from_yaml` is the quickest way to turn a YAML definition into a runnable Prefect
`@flow` without touching the config directory.  It accepts a YAML **string**, a `Path` to a
file, or a plain `dict`.


In [12]:
from pathlib import Path
from prefect import flow as prefect_flow, task as prefect_task
from genai_tk.workflow import flow_from_yaml

# ------------------------------------------------------------------ #
# Define a tiny @flow that we'll use as a step target                 #
# ------------------------------------------------------------------ #


@prefect_task
def reverse_string(s: str) -> str:
    return s[::-1]


@prefect_flow(name="reverse-words-flow")
def reverse_words_flow(words: list[str] | None = None) -> list[str]:
    """Reverse each word in the list."""
    if words is None:
        words = ["hello", "world"]
    futures = [reverse_string.submit(w) for w in words]
    return [f.result() for f in futures]


# Register the flow so flow_from_yaml can reference it by dotted path.
# In a real project the module would be importable — here we inject it:
import sys

this_module = type(sys)("notebook_flows")
this_module.reverse_words_flow = reverse_words_flow
sys.modules["notebook_flows"] = this_module

print("Registered flow:", reverse_words_flow)


Registered flow: <prefect.flows.Flow object at 0x7a68abcd0620>


In [13]:
INLINE_YAML = """
workflows:
  reverse_demo:
    description: "Demo: reverse a list of words using flow_from_yaml"
    run: notebook_flows.reverse_words_flow
    defaults:
      words:
        - hello
        - prefect
        - genai-tk
"""

# Parse YAML inline → get a Prefect @flow object
demo_flow = flow_from_yaml(INLINE_YAML)

print("Flow object:", demo_flow)
print("Flow name  :", demo_flow.name)

# Execute it — visible in the Prefect UI
results = demo_flow()
print("Results    :", results)


2026-05-30 21:04:31.386 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'reverse_demo': 1 steps


Flow object: <prefect.flows.Flow object at 0x7a68a9885dc0>
Flow name  : reverse_demo


21:04:31.518 | INFO    | Flow run 'sceptical-impala' - Beginning flow run 'sceptical-impala' for flow 'reverse_demo'

21:04:31.522 | INFO    | Flow run 'sceptical-impala' - View at http://127.0.0.1:4200/runs/flow-run/33b83f09-5879-4c20-a790-c7909b6a14b2

2026-05-30 21:04:31.534 | DEBUG    | genai_tk.workflow.prefect.flow_factory:_flow_body:404 - Submitted step 'run'


...

21:04:32.764 | INFO    | Flow run 'glossy-jackal' - Finished in state Completed()

21:04:32.771 | INFO    | Task run 'run-009' - Finished in state Completed()

2026-05-30 21:04:32.781 | INFO     | genai_tk.workflow.prefect.flow_factory:_flow_body:419 - Step 'run' completed


21:04:33.527 | INFO    | Flow run 'sceptical-impala' - Finished in state Completed()

Results    : {'run': ['olleh', 'tceferp', 'kt-ianeg']}


In [14]:
# You can also load from a file:
# demo_flow = flow_from_yaml(Path("config/workflows/my_pipeline.yaml"), workflow_name="my_pipeline")

# Or pass overrides:
# demo_flow = flow_from_yaml(INLINE_YAML, values={"words": ["override", "me"]})
# demo_flow()

# Signature: flow_from_yaml(source, *, workflow_name=None, values=None, max_workers=4)
help(flow_from_yaml)


Help on function flow_from_yaml in module genai_tk.workflow.prefect.flow_factory:

flow_from_yaml(source: 'str | Path | dict', *, workflow_name: 'str | None' = None, values: 'dict[str, Any] | None' = None, max_workers: 'int' = 4) -> "'Flow[[], dict[str, Any]]'"
    Parse a workflow YAML definition and return a ready-to-call Prefect ``@flow``.

    This is the simplest entry-point for programmatic or notebook use: define a
    workflow inline as YAML (or load one from a file), pass optional runtime values,
    and get back a standard Prefect ``Flow`` object that you can call, inspect, or
    hand to ``flow.serve()``.

    The returned flow uses the toolkit's Prefect server singleton — call
    :func:`~genai_tk.utils.prefect_server.prefect_server` (or ``cli prefect start``)
    before invoking it if ``prefect.auto_start`` is ``false``.

    Args:
        source: One of:
            - A YAML **string** containing a ``workflows:`` block.
            - A :class:`~pathlib.Path` to a YAML fil

## 7. `PrefectFlowFactory` — Compile and Run Config-Registered Workflows

`PrefectFlowFactory` is the lower-level class used by the CLI internally.  Use it when you
need more control (e.g. setting `max_workers`, getting the flow object without running it,
or serving a deployment).


In [15]:
from genai_tk.workflow import PrefectFlowFactory

# from_profile() resolves "name" or "name/preset" just like the CLI
factory = PrefectFlowFactory.from_profile(
    "markdownize",
    values={"base_dir": "/tmp/demo_in", "output_dir": "/tmp/demo_out"},
)

print("Compiled workflow :", factory.compiled.name)
print("Steps             :", [s.id for s in factory.compiled.steps])
print("Values            :", factory.compiled.values)

# Get the Prefect flow object without running it (useful for inspection)
flow_obj = factory.get()
print("\nPrefect Flow object:", flow_obj)
print("Flow name          :", flow_obj.name)


2026-05-30 21:05:54.693 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'markdownize': 1 steps


Compiled workflow : markdownize
Steps             : ['run']
Values            : {'pdf_converter': 'mistral', 'batch_size': 5, 'base_dir': '/tmp/demo_in', 'output_dir': '/tmp/demo_out', 'pathspecs': None}

Prefect Flow object: <prefect.flows.Flow object at 0x7a68746ff380>
Flow name          : markdownize


## 8. Multi-Step Pipeline Example

A three-step pipeline defined inline using `flow_from_yaml`.  Steps declare
`after:` dependencies; independent steps would run in parallel.


In [16]:
import sys
from prefect import flow as pf, task as pt

# --- Define three tiny flows as steps ---


@pt
def step_a_fn(msg: str) -> str:
    print(f"  [step_a] processing: {msg}")
    return msg.upper()


@pf(name="step-a")
def step_a(msg: str = "hello") -> str:
    return step_a_fn.submit(msg).result()


@pt
def step_b_fn(data: str) -> str:
    print(f"  [step_b] reversing: {data}")
    return data[::-1]


@pf(name="step-b")
def step_b(data: str = "") -> str:
    return step_b_fn.submit(data).result()


@pt
def step_c_fn(result: str) -> dict:
    print(f"  [step_c] finalising: {result}")
    return {"final": result, "length": len(result)}


@pf(name="step-c")
def step_c(result: str = "") -> dict:
    return step_c_fn.submit(result).result()


# Register them in sys.modules so flow_from_yaml can import them
mod = type(sys)("demo_pipeline_steps")
mod.step_a = step_a
mod.step_b = step_b
mod.step_c = step_c
sys.modules["demo_pipeline_steps"] = mod

print("Steps registered.")


Steps registered.


In [18]:
PIPELINE_YAML = """
workflows:
  demo_pipeline:
    description: "Three-step demo: uppercase → reverse → report"
    pipeline:
      - id: uppercase
        run: demo_pipeline_steps.step_a
        with:
          msg: hello_prefect

      - id: reverse
        run: demo_pipeline_steps.step_b
        after: [uppercase]
        with:
          data: "${steps.uppercase.result.}"

      - id: report
        run: demo_pipeline_steps.step_c
        after: [reverse]
        with:
          result: "${steps.reverse.result.}"
"""

# Note: ${steps.<id>.result.<field>} lets you wire outputs of one step
# into inputs of the next.  An empty field name ("result.") uses the whole result.

pipeline_flow = flow_from_yaml(PIPELINE_YAML)
print("Pipeline flow:", pipeline_flow.name)


2026-05-30 21:08:24.126 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'demo_pipeline': 3 steps


Pipeline flow: demo_pipeline


## 9. Fan-Out / Map Steps with `foreach:`

A `foreach:` step runs once **per item** in a collection, submitting all iterations as
concurrent Prefect tasks.  Use it to process a list of files, batches, IDs, etc. without
writing manual fan-out code.

**YAML syntax:**

```yaml
pipeline:
  - id: produce_items
    run: my_module.produce_flow          # returns a list

  - id: process_each
    run: my_module.process_flow
    after: [produce_items]
    foreach:
      from: "${steps.produce_items.result.}"  # source collection (whole result)
      as: item                                 # loop variable name
      # concurrency_limit: 4                  # optional cap on parallel tasks
    with:
      value: "${item}"                         # reference the current item
```

**Key rules:**
- `from:` must resolve to a `list` at runtime — use `${steps.<id>.result.}` or a literal.
- `as:` names the loop variable; reference it with `${item}` in `with:`.
- Results are collected as a **list** in `results[step_id]` — wire downstream steps with
  `${steps.<id>.result.}` to receive the full list.
- Iterations are submitted concurrently (limited by `max_workers` or `concurrency_limit`).

In [21]:
import sys
from prefect import flow as pf, task as pt

# --- Step 1: produce a list of items ---


@pt
def _produce(count: int) -> list[str]:
    return [f"item-{i}" for i in range(count)]


@pf(name="produce-items")
def produce_items(count: int = 5) -> list[str]:
    return _produce.submit(count).result()


# --- Step 2: process each item (runs once per item in the list) ---


@pt
def _process(value: str) -> dict:
    upper = value.upper()
    print(f"  [process] {value} → {upper}")
    return {"input": value, "output": upper}


@pf(name="process-item")
def process_item(value: str = "") -> dict:
    return _process.submit(value).result()


# Register in sys.modules so flow_from_yaml can import them
mod = type(sys)("foreach_demo_steps")
mod.produce_items = produce_items
mod.process_item = process_item
sys.modules["foreach_demo_steps"] = mod

print("Steps registered.")

Steps registered.


In [24]:
FOREACH_YAML = """
workflows:
  foreach_demo:
    description: "Fan-out demo: produce a list, then process each item in parallel"
    pipeline:
      - id: produce
        run: foreach_demo_steps.produce_items
        with:
          count: 4

      - id: process_each
        run: foreach_demo_steps.process_item
        after: [produce]
        foreach:
          from: "${steps.produce.result.}"
          as: item
        with:
          value: "${item}"
"""

foreach_flow = flow_from_yaml(FOREACH_YAML)
print("Flow:", foreach_flow.name)

results = foreach_flow()

print("\nProduced items :", results["produce"])
print("Processed items:", results["process_each"])

2026-05-30 21:25:29.005 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'foreach_demo': 2 steps


Flow: foreach_demo


21:25:29.164 | INFO    | Flow run 'invisible-wapiti' - Beginning flow run 'invisible-wapiti' for flow 'foreach_demo'

21:25:29.167 | INFO    | Flow run 'invisible-wapiti' - View at http://127.0.0.1:4200/runs/flow-run/ba30af54-347b-4042-9ac5-cdfe2d87a53f

2026-05-30 21:25:29.177 | DEBUG    | genai_tk.workflow.prefect.flow_factory:_flow_body:434 - Submitted step 'produce'


...

21:25:38.050 | INFO    | Flow run 'dexterous-antelope' - Finished in state Completed()

21:25:38.053 | INFO    | Task run 'process_each-9db' - Finished in state Completed()

2026-05-30 21:25:38.056 | INFO     | genai_tk.workflow.prefect.flow_factory:_flow_body:453 - Step 'process_each' completed


21:25:38.175 | INFO    | Flow run 'invisible-wapiti' - Finished in state Completed()


Produced items : ['item-0', 'item-1', 'item-2', 'item-3']
Processed items: [{'input': 'item-0', 'output': 'ITEM-0'}, {'input': 'item-1', 'output': 'ITEM-1'}, {'input': 'item-2', 'output': 'ITEM-2'}, {'input': 'item-3', 'output': 'ITEM-3'}]


In [ ]:
# --- Inspect the compiled ForeachSpec ---
import yaml
from genai_tk.workflow import ForeachSpec, WorkflowCompiler
from genai_tk.workflow.resolver import parse_workflows_from_dict

raw = yaml.safe_load(FOREACH_YAML)
candidates = parse_workflows_from_dict(raw["workflows"])
compiled = WorkflowCompiler().compile(list(candidates.values())[0], {})

print(f"{'Step':<16} {'Target':<36} {'Foreach'}")
print("-" * 72)
for step in compiled.steps:
    fe = step.foreach
    fe_str = f"from={fe.from_ref!r} as={fe.as_var!r}" if fe else "-"
    print(f"{step.id:<16} {step.invoke.target:<36} {fe_str}")

## 9. Inspecting Cache Status Programmatically

The Workflow Engine uses a **manifest cache** to skip steps whose inputs have not changed.
You can inspect cache freshness without running the CLI.


In [19]:
from genai_tk.workflow import ManifestCache, WorkflowCompiler
from genai_tk.workflow.prefect.flow_factory import (
    workflow_manifest_path,
    compute_step_fingerprint,
    _prepare_inputs,
)

# Resolve + compile the markdownize workflow (with dummy values for demo)
invocation = resolve_workflow_invocation(
    "markdownize",
    cli_overrides={"base_dir": "/tmp/demo_in", "output_dir": "/tmp/demo_out"},
)
compiled = WorkflowCompiler().compile(invocation.workflow, invocation.values)

# Load the manifest (returns empty manifest if file doesn't exist yet)
manifest_path = workflow_manifest_path(compiled.name)
manifest = ManifestCache.load(manifest_path)

print(f"Manifest path: {manifest_path}")
print(f"Existing records: {len(manifest.records)}\n")

# Inspect freshness of each step
print(f"{'Step':<20} {'Backend':<10} {'Fingerprint':<16} {'Status'}")
print("-" * 60)
for step in compiled.steps:
    step_inputs = _prepare_inputs(step.with_, {})
    fp = compute_step_fingerprint(step.id, step_inputs)
    backend = step.cache.backend
    is_fresh = manifest.is_fresh(step.id, fingerprint=fp) if backend != "none" else None
    status = "FRESH" if is_fresh else ("STALE/UNKNOWN" if is_fresh is not None else "no cache")
    print(f"{step.id:<20} {backend:<10} {fp[:12]:<16} {status}")


2026-05-30 21:08:39.132 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'markdownize': 1 steps


Manifest path: /home/tcl/prj/genai-tk/data/.workflow_manifests/markdownize/manifest.json
Existing records: 0

Step                 Backend    Fingerprint      Status
------------------------------------------------------------
run                  none       03f914dc6ad5     no cache


## 10. Serving a Workflow as a Prefect Deployment

A deployment registers a workflow with the Prefect server and blocks, waiting for runs
triggered from the UI, API, or a cron/interval schedule.

```python
# CLI equivalent (blocking):
#   cli workflow serve markdownize/docs --cron "0 2 * * *"

# Python equivalent (use in a dedicated process or background thread):
factory = PrefectFlowFactory.from_profile(
    "markdownize",
    values={"base_dir": "/data/pdfs", "output_dir": "/data/md"},
)
factory.serve(
    name="nightly-markdownize",
    cron="0 2 * * *",          # run at 02:00 every day
    # interval=3600,            # or: run every hour
    # tags=["nightly", "docs"], # optional Prefect tags
)
# ← blocks here; trigger runs from http://localhost:4200
```

To run a deployment in this notebook without blocking the kernel, start it in a background
thread (demo only — in production use a dedicated process or `cli workflow serve`):


In [20]:
import threading

# Build a factory from an inline YAML workflow
serve_factory = PrefectFlowFactory.from_profile(
    "markdownize",
    values={"base_dir": "/tmp/demo_in", "output_dir": "/tmp/demo_out"},
)

# Preview what the deployment would look like
print("Deployment name :", serve_factory.compiled.name)
print("Workflow steps  :", [s.id for s in serve_factory.compiled.steps])
print()
print("To start a real deployment, run in a terminal:")
print("  uv run cli workflow serve markdownize --base-dir /data --to /data/out")
print("  uv run cli workflow serve markdownize --cron '0 2 * * *'")
print(f"\nThen trigger runs from: http://localhost:4200")


2026-05-30 21:09:15.910 | DEBUG    | genai_tk.workflow.compiler:compile:68 - Compiled workflow 'markdownize': 1 steps


Deployment name : markdownize
Workflow steps  : ['run']

To start a real deployment, run in a terminal:
  uv run cli workflow serve markdownize --base-dir /data --to /data/out
  uv run cli workflow serve markdownize --cron '0 2 * * *'

Then trigger runs from: http://localhost:4200


## 11. Summary

| Task | Tool |
|------|------|
| Run any `@flow` | `server.ensure_running()` then call the flow directly |
| Load YAML inline | `flow_from_yaml(yaml_str)` → Prefect `@flow` |
| Load YAML from file | `flow_from_yaml(Path("config/workflows/my.yaml"))` |
| Run config workflow via Python | `PrefectFlowFactory.from_profile("name/preset").run()` |
| Run from CLI | `cli workflow run name[/preset] [--dry-run] [--set K=V]` |
| Fan-out over a list | `foreach: {from: "${steps.id.result.}", as: item}` + `with: {arg: "${item}"}` |
| List workflows | `cli workflow list` |
| Inspect a workflow | `cli workflow show name` |
| Validate all workflows | `cli workflow validate` |
| Serve as deployment | `cli workflow serve name[/preset] [--cron "..."]` |
| Inspect cache | `ManifestCache.load(path).is_fresh(step_id, fingerprint=fp)` |

**References:**
- [docs/prefect.md](../../docs/prefect.md) — managing the server, running `@flow` functions, `flow_from_yaml`
- [docs/workflows.md](../../docs/workflows.md) — full DSL reference, built-in workflows, how to create new ones
- [genai_tk/workflow/prefect/flow_factory.py](../../genai_tk/workflow/prefect/flow_factory.py) — `PrefectFlowFactory`, `flow_from_yaml`
- [genai_tk/utils/prefect_server.py](../../genai_tk/utils/prefect_server.py) — server singleton